In [1]:
import torch
import torchvision
import torch.utils.data
import torchvision.transforms.v2

transforms = torchvision.transforms.v2.Compose(
    [
        torchvision.transforms.v2.Resize(224),
        torchvision.transforms.v2.Grayscale(num_output_channels=3),
        torchvision.transforms.v2.ToImage(),
        torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
        torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
    ]
)
train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)

train_idxs = list(range(1000))
test_idxs = list(range(250))
train_y = train_ds.targets[train_idxs]
test_y = test_ds.targets[test_idxs]
train_ds = torch.utils.data.Subset(train_ds, indices=train_idxs)
test_ds = torch.utils.data.Subset(test_ds, indices=test_idxs)

In [2]:
import zigzag.nn
import zigzag.utils
import zigzag.pipelines

PARAMS = [
    zigzag.pipelines.Params(k_neighbors=2, dimension=3),
    zigzag.pipelines.Params(k_neighbors=3, dimension=3),
    zigzag.pipelines.Params(k_neighbors=4, dimension=3),
    zigzag.pipelines.Params(k_neighbors=5, dimension=3),
]
dumper = zigzag.utils.UniversalDumper("zigzag_results/testing/vit_b_16/subset")
# dumper.clear()

In [3]:
pretrained_dumper = dumper.make_subdumper("pretrained")
model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = torch.nn.Identity()

zigzag.pipelines.validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)
hidden_states = pretrained_dumper.execute(zigzag.nn.collect_hidden_states, "hidden_states", model, train_ds)
zigzag.pipelines.analyze(
    hidden_states, PARAMS, pretrained_dumper, class_labels=train_y, verbosity=zigzag.pipelines.Verbosity.ONLY_PROGRESSBAR
)

/usr/local/lib/python3.12/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Got the result from zigzag_results/testing/vit_b_16/subset/pretrained/train_embeddings.pt
Got the result from zigzag_results/testing/vit_b_16/subset/pretrained/test_embeddings.pt
Got the result from zigzag_results/testing/vit_b_16/subset/pretrained/train_head_history.csv
Got the result from zigzag_results/testing/vit_b_16/subset/pretrained/hidden_states


100%|██████████| 4/4 [01:01<00:00, 15.45s/it]


In [ ]:
finetuned_dumper = dumper.make_subdumper("finetuned")

model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = pretrained_dumper.get_dump("trained_head")

zigzag.pipelines.train_validate(model, train_ds, test_ds, dumper, learning_rate=1e-5)
hidden_states = dumper.execute(zigzag.nn.collect_hidden_states, "hidden_states", model, train_ds)
zigzag.pipelines.analyze(
    hidden_states, PARAMS, finetuned_dumper, class_labels=train_y, verbosity=zigzag.pipelines.Verbosity.ONLY_PROGRESSBAR
)

Got the result from zigzag_results/testing/vit_b_16/subset/pretrained/trained_head.pth


Training: 100%|██████████| 10/10 [07:38<00:00, 45.88s/it, Accuracy=0.972, AUC-ROC=0.999, Precision=0.971, Recall=0.972, F1-score=0.971, TOP-2 Accuracy=0.988, TOP-3 Accuracy=0.996, TOP-5 Accuracy=1, TOP-7 Accuracy=1, TOP-9 Accuracy=1]      


Saving the result to zigzag_results/testing/vit_b_16/subset/train_model_history.csv
Saving the result to zigzag_results/testing/vit_b_16/subset/trained_model.pth


Collect hidden states: 100%|██████████| 8/8 [00:16<00:00,  2.05s/it]


Saving the result to zigzag_results/testing/vit_b_16/subset/hidden_states


100%|██████████| 4/4 [01:02<00:00, 15.66s/it]
